In [167]:
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree
from datetime import datetime 
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import pandas as pd
import contextily as cnx
import matplotlib.pyplot as plt
from datetime import date
import re
from rasterstats import zonal_stats
import rasterio
import os
from tqdm import tqdm
from shapely.geometry import box
import math

<h2> Obtaining fire dates and information for each Footprint <h2>

In [168]:
# Input Files
RASTER_FILE_PATH = 'MapBiomas Fire/2019-2024/Burned_monthly_AoD_2019_2024-0000053760-0000080640.tif'
GEDI_FILE_PATH = 'Gedi_fire_GEDIdb.gpkg'


# The raster path string is needed by the zonal_stats function inside the loop
raster_path = RASTER_FILE_PATH

# Match the output name with the  

match = re.search(r'(\d+-\d+)\.tif$', RASTER_FILE_PATH)

if match:
    # If found, the ID is the first captured group (group 1)
    RASTER_ID = match.group(1)
else:
    # Fallback in case the pattern isn't found
    RASTER_ID = "UnknownID"

# Output File
OUTPUT_FILENAME = f"GEDI_Fire_Dates_{RASTER_ID}.gpkg"

print(f"File will be saved as: {OUTPUT_FILENAME}")

# Set the folder where you want to save the final GeoPackage
OUTPUT_FOLDER = "GEDI_Fire_Dates"

# Use os.path.join for platform compatibility (best practice)
OUTPUT_GPKG_PATH = os.path.join(OUTPUT_FOLDER, OUTPUT_FILENAME)

File will be saved as: GEDI_Fire_Dates_0000053760-0000080640.gpkg


In [169]:
#fire_scars_src = rasterio.open(RASTER_FILE_PATH)
#gedi_all_footprints = gpd.read_file(GEDI_FILE_PATH)

#print("Raster CRS:", fire_scars_src.crs)
#print("Raster bounds:", fire_scars_src.bounds)

#print("GEDI CRS:", gedi_all_footprints.crs)
#print("GEDI total bounds:", gedi_all_footprints.total_bounds)

#gedi_test = gedi_all_footprints.to_crs(fire_scars_src.crs)

#raster_polygon = box(
    #fire_scars_src.bounds.left,
    #fire_scars_src.bounds.bottom,
    #fire_scars_src.bounds.right,
    #fire_scars_src.bounds.top,
#)

#print("Intersects raster bbox:", gedi_test.intersects(raster_polygon).sum())
#print("Within raster bbox:", gedi_test.within(raster_polygon).sum())


In [170]:
print("Starting Spatial Filtering")

try:
    # 1. Load both inputs
    fire_scars_src = rasterio.open(RASTER_FILE_PATH)
    gedi_all_footprints = gpd.read_file(GEDI_FILE_PATH)
    
    initial_count = len(gedi_all_footprints)
    print(f"Loaded {initial_count} footprints before filtering.")

    # 2. Get Raster Bounding Box (BBOX) and create the clipping geometry
    raster_bounds = fire_scars_src.bounds
    
    # Use shapely.box to create a valid Polygon geometry from the BoundingBox coordinates
    raster_polygon = box(
        raster_bounds.left, 
        raster_bounds.bottom, 
        raster_bounds.right, 
        raster_bounds.top
    )

    # Create the GeoSeries for clipping, setting its CRS to the raster's CRS
    raster_extent_poly = gpd.GeoSeries(
        [raster_polygon], 
        crs=fire_scars_src.crs
    )

    # 3. Perform the Clipping (Filtering)
    gedi_for_zonal_stats = (
    gedi_all_footprints.to_crs(fire_scars_src.crs)
    .clip(raster_extent_poly)
    )

    # Repair invalid geometries but keep attributes
    gedi_for_zonal_stats["geometry"] = gedi_for_zonal_stats.geometry.buffer(0)

    # Remove empty geometries
    gedi_for_zonal_stats = gedi_for_zonal_stats[~gedi_for_zonal_stats.geometry.is_empty]

    # Ensure CRS is set correctly
    gedi_for_zonal_stats = gedi_for_zonal_stats.set_crs(fire_scars_src.crs, allow_override=True)
    final_count = len(gedi_for_zonal_stats)

    print(f"Filtered down to {final_count} footprints INSIDE the raster.")
    print(f"Successfully clipped out {initial_count - final_count} footprints.")

finally:
    # Ensure the raster file handle is closed immediately after getting its bounds
    if not fire_scars_src.closed:
        fire_scars_src.close()
    print("Raster source file closed.")
    print("--------------------------------")

Starting Spatial Filtering
Loaded 1027381 footprints before filtering.
Filtered down to 0 footprints INSIDE the raster.
Successfully clipped out 1027381 footprints.
Raster source file closed.
--------------------------------


In [171]:
gedi_for_zonal_stats.head()

,latitude,longitude,time,shot_number,digital_elevation_model,elev_lowestmode,cover,pai,l4_quality_flag,predictor_limit_flag,...,agbd_pi_lower,agbd_pi_upper,agbd_se,wsci,rh_50,rh_90,rh_98,tile_id,gedi_point_wkt,geometry


In [172]:
# Corrected function to extract fire dates with None handling

def extract_fire_dates(fire_scars_src, GEDI_shots_gdf): 
    with fire_scars_src as src: 
        gdf_polygons = GEDI_shots_gdf.copy()

        band_years = [int(re.search(r"(\d{4})", d).group(1)) for d in src.descriptions]

        all_fire_dates = []
        fire_counts = []

        print(f"Starting Zonal Stats for {len(gdf_polygons)} footprints...")
        
        for poly in tqdm(gdf_polygons.geometry, desc="Extracting Fire Info"):
            dates = []
            for b in range(1, src.count + 1):
                stats = zonal_stats(poly, raster_path, stats="majority", band=b)
                month = stats[0].get("majority", None)
                if month is not None and month > 0: 
                    year = band_years[b - 1]
                    dates.append(date(year, int(month), 1))
            all_fire_dates.append(dates)
            fire_counts.append(len(dates))
        
        gdf_polygons["fire_dates"] = all_fire_dates
        gdf_polygons["fire_count"] = fire_counts
        gdf_polygons["first_fire_date"] = [d[0] if d else None for d in all_fire_dates]
        gdf_polygons["last_fire_date"]  = [d[-1] if d else None for d in all_fire_dates]

        # QGIS-friendly conversions
        gdf_polygons["fire_dates_str"] = [
            ",".join(d.strftime("%Y-%m-%d") for d in dates) if dates else ""
            for dates in gdf_polygons["fire_dates"]
        ]
        import pandas as pd
        gdf_polygons["first_fire_date"] = pd.to_datetime(gdf_polygons["first_fire_date"])
        gdf_polygons["last_fire_date"]  = pd.to_datetime(gdf_polygons["last_fire_date"])

        # Optional: drop raw list column
        gdf_polygons = gdf_polygons.drop(columns=["fire_dates"])

        return gdf_polygons



In [173]:
# Configuration for batching
BATCH_SIZE = 50000  # adjust to suit memory/speed

total = len(gedi_for_zonal_stats)
num_batches = math.ceil(total / BATCH_SIZE)

print("\n" + "="*60)
print(f"Processing {total} footprints in {num_batches} batches of {BATCH_SIZE}...")
print(f"Tile ID: {RASTER_ID}")
print("="*60)

# Re-open the raster source for processing
fire_scars_src = rasterio.open(RASTER_FILE_PATH)

try:
    for i in range(num_batches):
        start = i * BATCH_SIZE
        end = min((i + 1) * BATCH_SIZE, total)
        
        # Build batch filename with tile ID + batch number
        batch_filename = f"GEDI_Fire_Dates_{RASTER_ID}_batch{i+1}.gpkg"
        batch_path = os.path.join(OUTPUT_FOLDER, batch_filename)

        # Safe restart: skip if this batch file already exists
        if os.path.exists(batch_path):
            print(f"Skipping batch {i+1}/{num_batches} (already exists): {batch_path}")
            continue

        print(f"\n--- Batch {i+1}/{num_batches}: footprints {start} to {end} ---")
        batch_gdf = gedi_for_zonal_stats.iloc[start:end]

        # Run extraction for this batch
        batch_result = extract_fire_dates(fire_scars_src, batch_gdf)

        # Save incrementally
        batch_result.to_file(batch_path, driver="GPKG")
        print(f"✅ Saved batch {i+1} to {batch_path}")

finally:
    if not fire_scars_src.closed:
        fire_scars_src.close()
    print("Raster source file closed.")



Processing 0 footprints in 0 batches of 50000...
Tile ID: 0000053760-0000080640
Raster source file closed.
